# Suite CONF — Cross-implementation conformance

Every implementation builds the same corpus (`mbse/Patterns/Conformance/Corpus.py`, mirrored statement for statement
in TypeScript) and commits its snapshots under `conformance/<implementation>/`. This suite checks this implementation's
files are current, that its JSON is byte-identical to every other implementation's, and that it reads every other
implementation's JSON and YAML back to the same sets. Regenerate with `uv run python -m mbse.Patterns.Conformance.write`.

In [ ]:
from pathlib import Path

from mbse.Patterns.Conformance.Corpus import CASES, build
from mbse.Patterns.Conformance.write import render
from mbse.Schemas.Framework import JSON, Plain, YAML

ROOT = Path("../../conformance")
OWN = "python3"
OTHERS = sorted(p.name for p in ROOT.iterdir() if p.is_dir() and p.name != OWN)
assert OTHERS == ["typescript5"], OTHERS

corpus = build()

## CONF-01 · This implementation's committed files are current

In [ ]:
for name, text in render(corpus).items():
    assert (ROOT / OWN / name).read_text(encoding="utf-8") == text, f"{name} is stale: regenerate the corpus"

## CONF-02 · Every implementation has the same cases, and JSON is byte-identical across implementations

In [ ]:
for name in [OWN, *OTHERS]:
    assert sorted(p.name for p in (ROOT / name).iterdir()) == sorted(f"{c}.{e}" for c in CASES for e in ("json", "yaml")), name
for other in OTHERS:
    for case in CASES:
        assert (ROOT / OWN / f"{case}.json").read_bytes() == (ROOT / other / f"{case}.json").read_bytes(), f"{case}.json differs from {other}"

## CONF-03 · This implementation reads every implementation's JSON and YAML back to the same sets

In [ ]:
for other in [OWN, *OTHERS]:
    for case, (schema, root, store) in corpus.items():
        expected = Plain.ToPlain(store).Reachable(schema, root)
        from_json = JSON.FromJSON(store).Reachable(schema, (ROOT / other / f"{case}.json").read_text(encoding="utf-8"))
        from_yaml = YAML.FromYAML(store).Reachable(schema, (ROOT / other / f"{case}.yaml").read_text(encoding="utf-8"))
        assert Plain.ToPlain(store).Reachable(schema, from_json) == expected, (other, case, "json")
        assert Plain.ToPlain(store).Reachable(schema, from_yaml) == expected, (other, case, "yaml")